In [ ]:
from pathlib import Path
from io import BytesIO
import json, csv, hashlib, math

PACKAGE_ROOT = next((p for p in (Path.cwd(), Path.cwd().parent)
                     if (p / "data").is_dir() and (p / "notebooks").is_dir()), None)
if PACKAGE_ROOT is None:
    raise FileNotFoundError("Open this notebook from the package root or notebooks directory.")

SAVE_PDF = False

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.stats import t
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from IPython.display import display, Image
plt.rcdefaults()

def load_npz(relative_path):
    with np.load(PACKAGE_ROOT / relative_path, allow_pickle=False) as archive:
        return {key: archive[key].copy() for key in archive.files}



def export_figure(figure, figure_id, tight=False):
    options = {"bbox_inches": "tight"} if tight else {}
    preview = BytesIO()
    figure.savefig(preview, format="png", dpi=180, **options)
    display(Image(data=preview.getvalue()))
    if SAVE_PDF:
        output = PACKAGE_ROOT / "outputs"
        output.mkdir(exist_ok=True)
        figure.savefig(output / f"{figure_id}.pdf",
                       metadata={"CreationDate": None, "ModDate": None}, **options)
        print(f"Saved outputs/{figure_id}.pdf")


In [ ]:
CONFIGS = ('iris_noiseless', 'iris_pauli_y005', 'fashion_interleaved4', 'fashion_interleaved8')
TITLES = ('Iris: noiseless', r'Iris: encoder Pauli-Y, $p_y=0.05$',
          'Fashion-MNIST: interleaved / 4 qubits', 'Fashion-MNIST: interleaved / 8 qubits')
data=load_npz("data/FigS9/metrics.npz")
for name in CONFIGS:
    for split in ("train","test"):
        assert data[name+"_"+split].shape==(20,501,3)

In [ ]:
plt.rcParams.update({'font.family': 'DejaVu Serif', 'font.size': 9, 'axes.titlesize': 9,
  'axes.labelsize': 9, 'legend.fontsize': 9, 'xtick.labelsize': 8.5, 'ytick.labelsize': 8.5,
  'pdf.fonttype': 42, 'ps.fonttype': 42, 'axes.spines.top': False, 'axes.spines.right': False,
  'lines.linewidth': 1.25})
fig, axes = plt.subplots(4, 2, figsize=(7.0, 7.3), sharex=True)
colors = {'train': '#2166AC', 'test': '#D55E00'}
upper = max(float((data[n+'_'+s][:, :, 0].mean(0)+data[n+'_'+s][:, :, 0].std(0, ddof=1)).max())
            for n in CONFIGS for s in ('train', 'test'))
ylim = max(.70, np.ceil(upper*10)/10)
for row, (name, title) in enumerate(zip(CONFIGS, TITLES)):
    for col, k in enumerate((0, 2)):
        ax = axes[row, col]
        for split in ('train', 'test'):
            a = data[name+'_'+split][:, :, k]*(100 if k == 2 else 1)
            mu, sd = a.mean(0), a.std(0, ddof=1)
            ax.plot(data['updates'], mu, color=colors[split], ls='-' if split == 'train' else '--', label=split.title())
            ax.fill_between(data['updates'], np.maximum(0, mu-sd), np.minimum(100, mu+sd) if k == 2 else mu+sd,
                            color=colors[split], alpha=.15, lw=0)
        ax.set_title(f'({chr(97+2*row+col)}) '+title, loc='left', pad=5)
        ax.set_xlim(0, 500); ax.set_xticks([0, 100, 200, 300, 400, 500])
        ax.set_ylabel('Brier loss' if col == 0 else 'Accuracy (%)'); ax.grid(axis='y', alpha=.18, lw=.5)
        if col == 0: ax.set_ylim(0, ylim); ax.set_yticks(np.arange(0, ylim+.001, .2))
        else: ax.set_ylim(0, 105); ax.set_yticks([0, 25, 50, 75, 100])
        if row == 3: ax.set_xlabel('Completed GD updates')
fig.legend(*axes[0, 0].get_legend_handles_labels(), loc='upper center', ncol=2,
           frameon=False, bbox_to_anchor=(.52, 1.0))
fig.subplots_adjust(left=.095, right=.985, bottom=.075, top=.93, hspace=.42, wspace=.29)

export_figure(fig,"FigS9")
plt.close(fig)